In [ ]:
"""
Rice Leaf Disease Classification - Training Script
===================================================
Hybrid CNN (ResNet + DenseNet + SE) for 12-class classification.

Dataset: 3,656 train / 455 val / 464 test images
Classes: 12
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix
)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 120

TRAIN_DIR = "./rice_leaf_split/train"
VAL_DIR   = "./rice_leaf_split/validation"
TEST_DIR  = "./rice_leaf_split/test"

MODEL_SAVE_PATH = "./best_hybrid_model.h5"

In [ ]:
# ============================================================
# DATA GENERATORS
# ============================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.10,
    width_shift_range=0.08,
    height_shift_range=0.08,
    horizontal_flip=True,
    brightness_range=[0.88, 1.12],
    fill_mode='reflect'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
NUM_CLASSES = len(class_names)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)
print(f"Classes ({NUM_CLASSES}): {class_names}")
print(f"Train:      {train_generator.samples}")
print(f"Validation: {val_generator.samples}")
print(f"Test:       {test_generator.samples}")
print("=" * 70)

In [ ]:
# ============================================================
# MODEL ARCHITECTURE
# ============================================================
def residual_block(x, filters):
    shortcut = Conv2D(filters, 1, padding="same")(x)
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(0.1)(x)
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = LeakyReLU(0.1)(x)
    return x

def dense_block(x, growth_rate=32):
    y = Conv2D(growth_rate, 3, padding="same", activation="relu")(x)
    x = Concatenate()([x, y])
    y = Conv2D(growth_rate, 3, padding="same", activation="relu")(x)
    x = Concatenate()([x, y])
    return x

def se_block(x):
    filters = x.shape[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(filters // 4, activation="relu")(se)
    se = Dense(filters, activation="sigmoid")(se)
    se = Reshape((1, 1, filters))(se)
    return Multiply()([x, se])

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = Conv2D(64, 3, padding="same")(inputs)
x = BatchNormalization()(x)
x = LeakyReLU(0.1)(x)

x = residual_block(x, 64)
x = MaxPooling2D()(x)

x = dense_block(x)
x = se_block(x)
x = MaxPooling2D()(x)

x = residual_block(x, 128)
x = MaxPooling2D()(x)

x = dense_block(x)
x = se_block(x)
x = MaxPooling2D()(x)

x = residual_block(x, 256)

x = GlobalAveragePooling2D()(x)
x = Dense(512)(x)
x = BatchNormalization()(x)
x = LeakyReLU(0.1)(x)
x = Dropout(0.5)(x)

x = Dense(256)(x)
x = LeakyReLU(0.1)(x)
x = Dropout(0.3)(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs, outputs)

optimizer = AdamW(learning_rate=0.0001, weight_decay=1e-4)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# ============================================================
# CLASS WEIGHTS
# ============================================================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights))

print("\nClass weights:")
for i, cls in enumerate(class_names):
    print(f"  {cls}: {class_weights[i]:.4f}")

In [ ]:
# ============================================================
# CALLBACKS
# ============================================================
callbacks = [
    ReduceLROnPlateau(
        monitor='val_loss',
        patience=5,
        factor=0.3,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# ============================================================
# TRAINING
# ============================================================
print("\n" + "=" * 70)
print("STARTING TRAINING")
print("=" * 70)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# SAVE TRAINING HISTORY
# ============================================================
import json

history_dict = {
    'accuracy': [float(x) for x in history.history['accuracy']],
    'val_accuracy': [float(x) for x in history.history['val_accuracy']],
    'loss': [float(x) for x in history.history['loss']],
    'val_loss': [float(x) for x in history.history['val_loss']],
}

with open('./training_history.json', 'w') as f:
    json.dump(history_dict, f)

print("Training history saved to training_history.json")

In [ ]:
"""
Evaluation Script
=================
Evaluate trained model on validation and test sets.
Generate classification reports, confusion matrices, and plots.
"""

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ============================================================
# CONFIGURATION
# ============================================================
IMG_SIZE = 224
BATCH_SIZE = 32
MODEL_PATH = "./best_hybrid_model.h5"

VAL_DIR = "./rice_leaf_split/validation"
TEST_DIR = "./rice_leaf_split/test"

# ============================================================
# LOAD MODEL
# ============================================================
model = load_model(MODEL_PATH)

# ============================================================
# DATA GENERATORS (no augmentation)
# ============================================================
datagen = ImageDataGenerator(rescale=1./255)

val_generator = datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(val_generator.class_indices.keys())

# ============================================================
# EVALUATE ON VALIDATION SET
# ============================================================
print("=" * 70)
print("VALIDATION SET EVALUATION")
print("=" * 70)

val_pred = model.predict(val_generator, verbose=1)
val_pred_classes = np.argmax(val_pred, axis=1)
val_true = val_generator.classes

val_accuracy = accuracy_score(val_true, val_pred_classes) * 100
print(f"\nValidation Accuracy: {val_accuracy:.2f}%")
print("\nClassification Report (Validation):")
print(classification_report(val_true, val_pred_classes, target_names=class_names))

# ============================================================
# EVALUATE ON TEST SET
# ============================================================
print("\n" + "=" * 70)
print("TEST SET EVALUATION")
print("=" * 70)

test_pred = model.predict(test_generator, verbose=1)
test_pred_classes = np.argmax(test_pred, axis=1)
test_true = test_generator.classes

test_accuracy = accuracy_score(test_true, test_pred_classes) * 100
print(f"\nTest Accuracy: {test_accuracy:.2f}%")
print("\nClassification Report (Test):")
print(classification_report(test_true, test_pred_classes, target_names=class_names))


In [ ]:
# ============================================================
# CONFUSION MATRIX (TEST SET)
# ============================================================
cm = confusion_matrix(test_true, test_pred_classes)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names
)
plt.title('Confusion Matrix - Test Set', fontsize=14)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_test.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nConfusion matrix saved: confusion_matrix_test.png")

# ============================================================
# SAVE RESULTS
# ============================================================
results = {
    'val_accuracy': float(val_accuracy),
    'test_accuracy': float(test_accuracy),
    'val_report': classification_report(val_true, val_pred_classes,
                                         target_names=class_names, output_dict=True),
    'test_report': classification_report(test_true, test_pred_classes,
                                          target_names=class_names, output_dict=True),
}

with open('./evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved: evaluation_results.json")


In [ ]:
"""
Plot Training Curves
====================
"""

import json
import matplotlib.pyplot as plt

with open('./training_history.json', 'r') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history['loss'], label='Train', linewidth=2)
axes[1].plot(history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: training_curves.png")